In [ ]:
!pip install llama-index llama-index-llms-openai llama-index-embeddings-openai

INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
!pip install llama-index llama-index-llms-huggingface llama-index-embeddings-huggingface transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of huggingface-hub[inference] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of huggingface-hub[inference] to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longe

In [ ]:
import os

# 1. Create the directory
folder_name = "movie_data"
if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    print(f"Directory '{folder_name}' created!")

# 2. Create a sample movie file to test the system
sample_text = """
Inception is a 2010 science fiction action film written and directed by Christopher Nolan.
The film stars Leonardo DiCaprio as a professional thief who steals information by infiltrating the subconscious of his targets.
It won four Academy Awards and is known for its complex plot involving dreams within dreams.
"""

with open(f"{folder_name}/inception.txt", "w") as f:
    f.write(sample_text)
    print("Sample movie data 'inception.txt' created!")

Directory 'movie_data' created!
Sample movie data 'inception.txt' created!


In [ ]:
import torch
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from transformers import BitsAndBytesConfig

# 1. SETUP THE EMBEDDING MODEL (The "Librarian")
# This runs locally and is 100% free.
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

# 2. SETUP THE LLM (The "Writer")
# We use 4-bit quantization to make sure it doesn't crash Colab.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# Replace the LLM section with this:
Settings.llm = HuggingFaceLLM(
    model_name="microsoft/Phi-3-mini-4k-instruct",
    tokenizer_name="microsoft/Phi-3-mini-4k-instruct",
    context_window=4096,
    max_new_tokens=256,
    generate_kwargs={"temperature": 0.0},
    model_kwargs={"quantization_config": quantization_config},
    device_map="auto",
)

# 3. LOAD DATA AND BUILD INDEX
# Make sure your 'movie_data' folder exists as we did in the last step!
documents = SimpleDirectoryReader("./movie_data").load_data()
index = VectorStoreIndex.from_documents(documents)

# 4. QUERY WITH R2C2 REQUIREMENTS
query_engine = index.as_query_engine(similarity_top_k=3)

def run_ntcir_query(question):
    prompt = f"Based on the context, answer the question. Provide a CONFIDENCE score (0-100) and bulleted NUGGETS. \nQuestion: {question}"
    response = query_engine.query(prompt)
    return response

# Test
print(run_ntcir_query("Tell me about the movie Inception."))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Confidence score: 95
- Inception is a 2010 science fiction action film.
- Directed by Christopher Nolan.
- Starring Leonardo DiCaprio.
- The plot revolves around a professional thief who steals information by infiltrating the subconscious of his targets.
- Known for its complex plot involving dreams within dreams.
- Won four Academy Awards.





In [ ]:
import torch
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from transformers import BitsAndBytesConfig

# 1. Use an ungated embedding model (Fast & Free)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# 2. Setup Quantization (Crucial for Colab T4 GPU memory)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

# 3. Setup LLM (Using Phi-3 to avoid gating issues)
Settings.llm = HuggingFaceLLM(
    model_name="microsoft/Phi-3-mini-4k-instruct",
    tokenizer_name="microsoft/Phi-3-mini-4k-instruct",
    model_kwargs={"quantization_config": quantization_config},
    device_map="auto",
)

# 4. Load Data (Make sure your 'movie_data' folder has .txt files!)
documents = SimpleDirectoryReader("./movie_data").load_data()
index = VectorStoreIndex.from_documents(documents)

# 5. R2C2 Task Specific Query
from llama_index.core import PromptTemplate # Add this import at the top

def ask_ntcir(question):
    query_engine = index.as_query_engine(similarity_top_k=3)

    # 1. Define the string
    prompt_str = (
        "<|system|>\n"
        "You are a movie researcher for NTCIR-19. Answer ONLY using the context. "
        "If the answer isn't in the context, say 'I don't know' and set CONFIDENCE to 0.\n<|end|>\n"
        "<|user|>\n"
        "Context:\n{context_str}\n" # Note: use single curly braces {context_str} here
        f"Question: {question}\n"
        "Format your response as:\n"
        "ANSWER: [direct answer]\n"
        "CONFIDENCE: [0-100]\n"
        "NUGGETS: [bullet points]\n<|end|>\n"
        "<|assistant|>\n"
    )

    # 2. CONVERT THE STRING TO A PROMPT OBJECT (This is the fix!)
    text_qa_template = PromptTemplate(prompt_str)

    # 3. Update the query engine using the object
    query_engine.update_prompts(
        {"response_synthesizer:text_qa_template": text_qa_template}
    )

    response = query_engine.query(question)
    return response

# Execution
print(ask_ntcir("What is the main plot of the movie in the database?"))

# Execution
print(ask_ntcir("What is the main plot of the movie in the database?"))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

ANSWER: The main plot of the movie involves a professional thief who steals information by infiltrating the subconscious of his targets.
CONFIDENCE: 100
NUGGETS:
- The film is titled "Inception."
- It is a science fiction action film.
- Directed by Christopher Nolan.
- Released in 2010.
- Starring Leonardo DiCaprio.
- Won four Academy Awards.
- Known for its complex plot involving dreams within dreams.
ANSWER: The main plot of the movie involves a professional thief who steals information by infiltrating the subconscious of his targets.
CONFIDENCE: 100
NUGGETS:
- The film is titled "Inception."
- It is a science fiction action film.
- Directed by Christopher Nolan.
- Released in 2010.
- Starring Leonardo DiCaprio.
- Won four Academy Awards.
- Known for its complex plot involving dreams within dreams.


In [ ]:
import pandas as pd

def run_batch_evaluation(questions_list):
    all_results = []

    print(f"Starting evaluation for {len(questions_list)} questions...")

    for i, q in enumerate(questions_list):
        print(f"Processing Q{i+1}: {q}")
        response = ask_ntcir(q)

        # Store the raw text response
        all_results.append({
            "question": q,
            "system_output": str(response)
        })

    # Save to a CSV for your portfolio/submission
    df = pd.DataFrame(all_results)
    df.to_csv("ntcir_r2c2_results.csv", index=False)
    print("Done! Results saved to 'ntcir_r2c2_results.csv'")
    return df

# Example Usage:
test_questions = [
    "Who directed the movie Inception?",
    "When was Inception released?",
    "What awards did Inception win?"
]

results_df = run_batch_evaluation(test_questions)

Starting evaluation for 3 questions...
Processing Q1: Who directed the movie Inception?
Processing Q2: When was Inception released?
Processing Q3: What awards did Inception win?
Done! Results saved to 'ntcir_r2c2_results.csv'


In [ ]:
import pandas as pd
import json
import os

def prepare_movie_corpus(file_path, output_dir="./movie_data"):
    # Create the directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created directory: {output_dir}")

    # Detect file type and load
    if file_path.endswith('.csv'):
        df = pd.read_csv(file_path)
        # Convert to a list of dictionaries for easier processing
        data = df.to_dict(orient='records')
    elif file_path.endswith('.json'):
        with open(file_path, 'r') as f:
            data = json.load(f)
    else:
        print("Unsupported file format. Please use .csv or .json")
        return

    print(f"Processing {len(data)} movie records...")

    for i, record in enumerate(data):
        # We assume your file has columns like 'title' and 'content' or 'plot'
        # Adjust these keys based on your specific NTCIR file!
        title = record.get('title', f"movie_{i}")
        content = record.get('content', record.get('plot', record.get('text', "")))

        if not content:
            continue

        # Clean title to use as a filename
        safe_title = "".join([c for c in str(title) if c.isalnum() or c==' ']).rstrip()
        filename = f"{output_dir}/{safe_title.replace(' ', '_')}.txt"

        with open(filename, "w", encoding="utf-8") as f:
            f.write(f"Title: {title}\n\n{content}")

    print(f"Finished! All movies are now in {output_dir}/")

# --- HOW TO USE ---
# 1. Upload your 'ntcir_movies.csv' to Colab
# 2. Run this:
# prepare_movie_corpus("ntcir_movies.csv")

In [ ]:
from llama_index.core import Settings

# This limits how much text is processed at once to prevent GPU crashes
Settings.chunk_size = 512  # Smaller chunks = more precise retrieval
Settings.chunk_overlap = 50 # Overlap ensures context isn't cut in half

In [ ]:
import re

def parse_model_output(response_text):
    # Use Regular Expressions to find the confidence number
    confidence_match = re.search(r"CONFIDENCE:\s*(\d+)", response_text)
    confidence = int(confidence_match.group(1)) if confidence_match else 0

    # Extract Nuggets (lines starting with - or 1.)
    nuggets = re.findall(r"-\s*(.*)", response_text)

    return {
        "confidence": confidence,
        "nuggets": nuggets[:10] # NTCIR limit is 10 nuggets
    }

In [ ]:
from datasets import load_dataset
import os
import re

# 1. Load a reliable dataset (Wikipedia Movies)
print("Downloading 100 movie records from Hugging Face...")
dataset = load_dataset("Coder-Dragon/wikipedia-movies", split="train[:100]")

# 2. Create the directory
output_dir = "./movie_data"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 3. Save each movie as a separate .txt file
for i, item in enumerate(dataset):
    title = item['Title']
    plot = item['Plot']

    # Sanitize title for filename: remove non-alphanumeric chars
    safe_title = re.sub(r'[^\w\s]', '', title).strip().replace(' ', '_')
    filename = f"{output_dir}/{safe_title}.txt"

    with open(filename, "w", encoding="utf-8") as f:
        # Saving in a clear format for LlamaIndex to parse
        f.write(f"MOVIE TITLE: {title}\n")
        f.write(f"STORY PLOT: {plot}")

print(f"Success! Your library now contains {len(os.listdir(output_dir))} movie files.")

README.md: 0.00B [00:00, ?B/s]

plots.csv:   0%|          | 0.00/75.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30671 [00:00<?, ? examples/s]

Success! Your library now contains 103 movie files.


In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext, load_index_from_storage

# Load the new documents
documents = SimpleDirectoryReader("./movie_data").load_data()

# Build the index
index = VectorStoreIndex.from_documents(documents)

# Save the index to a folder named 'storage'
index.storage_context.persist(persist_dir="./storage")
print("Index built and saved to ./storage")

Index built and saved to ./storage


In [ ]:
# Check what titles you have
print("Available Movies (First 5):", os.listdir("./movie_data")[:5])

# Ask about a specific one (e.g., Alice in Wonderland is usually in this dataset)
response = ask_ntcir("Who is the main character in Alice in Wonderland?")
print(response)

Available Movies (First 5): ['Pocahontas.txt', 'From_Leadville_to_Aspen_A_HoldUp_in_the_Rockies.txt', 'Baseball_and_Bloomers.txt', 'Kathleen_Mavourneen.txt', 'The_Warrens_of_Virginia.txt']
ANSWER: Alice
CONFIDENCE: 100
NUGGETS:
- Alice follows a large white rabbit down a "Rabbit-hole".
- She finds a tiny door and a bottle labeled "Drink me".
- Alice eats something labeled "Eat me" and grows larger.
- She enters a kitchen and persuades the woman to give her the child.
- Alice is invited to join the "ROYAL PROCESSION": a parade of marching playing cards.
- She unintentionally offends the Queen, leading to her escape.
- Alice wakes up and realizes it was all a dream.


In [ ]:
import re

def generate_ntcir_run(questions_dict, run_name="MY_TEAM_AC_1"):
    """
    questions_dict: { "0001": "What is the plot of Alice in Wonderland?", ... }
    """
    run_output = []

    for topic_id, question in questions_dict.items():
        print(f"Processing Topic {topic_id}...")
        raw_response = ask_ntcir(question) # Uses your existing function
        response_text = str(raw_response)

        # 1. Extract Answer (find text after 'ANSWER:')
        answer_match = re.search(r"ANSWER:\s*(.*)", response_text, re.IGNORECASE)
        answer = answer_match.group(1).strip() if answer_match else "I don't know"

        # 2. Extract Confidence (find integer after 'CONFIDENCE:')
        conf_match = re.search(r"CONFIDENCE:\s*(\d+)", response_text)
        confidence = conf_match.group(1) if conf_match else "0"

        # 3. Extract Nuggets (lines starting with '-' or 'nugget')
        nuggets = re.findall(r"-\s*(.*)", response_text)

        # Build the NTCIR block
        run_output.append(f"<{topic_id}>")
        run_output.append(f"{answer};{confidence}")

        # R2C2 allows up to 10 nuggets per topic
        for i, nugget in enumerate(nuggets[:10]):
            # PassageKey is usually 'RunName;Rank' (using 'LOCAL;1' for now)
            run_output.append(f"{i+1};LOCAL;1;{nugget.strip()}")

        run_output.append(f"</{topic_id}>")

    # Save to file
    with open(f"{run_name}.txt", "w") as f:
        f.write("\n".join(run_output))

    print(f"Submission file created: {run_name}.txt")

# Example for your run
my_topics = {
    "0001": "What is the plot of Alice in Wonderland?",
    "0002": "Who is the main character in Pocahontas?"
}

generate_ntcir_run(my_topics)

Processing Topic 0001...
Processing Topic 0002...
Submission file created: MY_TEAM_AC_1.txt


In [ ]:
def test_system_modesty():
    print("--- 🛡️ Starting Modesty Stress Test ---")

    # 1. A question about a movie definitely NOT in your small sample
    tricky_question = "What happens in the 2024 movie 'Deadpool & Wolverine'?"

    print(f"Testing with OOD Question: {tricky_question}")
    response = ask_ntcir(tricky_question)

    # Parse the result
    response_text = str(response)
    conf_match = re.search(r"CONFIDENCE:\s*(\d+)", response_text)
    confidence = int(conf_match.group(1)) if conf_match else 100

    print(f"\nModel Output:\n{response_text}")

    if confidence < 30:

        print("\n✅ TEST PASSED: The system is modest. It recognized it lacked context.")
    else:
        print("\n❌ TEST FAILED: The system is 'overconfident'. You may need to tune the prompt.")

# Run the test
test_system_modesty()

--- 🛡️ Starting Modesty Stress Test ---
Testing with OOD Question: What happens in the 2024 movie 'Deadpool & Wolverine'?

Model Output:
ANSWER: I don't know
CONFIDENCE: 0
NUGGETS:
- The context provided does not contain information about the movie 'Deadpool & Wolverine'.
- The context only includes details about the plots of 'The Perils of Pauline', 'Mabel's Strange Predicament', and 'Between Showers'.
- The movie 'Deadpool & Wolverine' is not mentioned in the provided file paths or story plots.

✅ TEST PASSED: The system is modest. It recognized it lacked context.
